# 0. 데이터 생성

원천 `data/cache/els3_dataset.parquet` + raw `SCHD_INFO`(전체 STRK 스케줄)를 읽어 모델링 데이터셋 **3종**을 `data/` 에 **CSV**로 생성합니다.
컬럼명은 raw DART와 직접 비교되도록 raw 이름을 활용(예: `fair → FAIR_VALUE/ISU_PRC_DETAIL`, `B → BARR_1/100`).
정통 DeepONet 재설계: **branch = 시장상태(바스켓 vol·corr + 수익률곡선), trunk = 계약(전체 STRK 스케줄 strk_0..11 + 배리어 + 쿠폰 + 만기)**.

| 파일 | 대상 모델 | 핵심 컬럼 |
|---|---|---|
| `data/ml.csv` | tabular 벤치마크 + 하이브리드 stage-2 마진모델 | BASE+REG+CAT + FAIR + MC + recent_margin |
| `data/pi_deeponet.csv` | **PI-DeepONet 하이브리드** (물리) | vol·corr(7) + 곡선 u0..u9 + 계약(strk_0..11, BARR, coupon, TENOR) + r + FAIR + MC + recent_margin |
| `data/deeponet.csv` | **DeepONet-Curve 직접/하이브리드** (CNN) | 곡선 u0..u9 + vol·corr(7) + 계약(strk_0..11, BARR, coupon, TENOR) + r + FAIR + MC + recent_margin |

In [1]:
from util import utils, file_manager as fm
from module import data
import pandas as pd

cfg = utils.load_config()
info = data.build_datasets()
print("rows:", {k: info[k] for k in ("ml", "pi_deeponet", "deeponet")})

rows: {'ml': 23151, 'pi_deeponet': 23151, 'deeponet': 23151}


In [2]:
# 생성 데이터셋 확인 (CSV, raw 기반 컬럼명)
for name in ("ml", "pi_deeponet", "deeponet"):
    df = pd.read_csv(fm.dataset(name), encoding="utf-8-sig")
    print(f"\n=== {name}.csv | shape {df.shape} ===")
    print("columns:", list(df.columns))
    display(df.head(3))


=== ml.csv | shape (23151, 40) ===
columns: ['sig1', 'sig2', 'sig3', 'rho12', 'rho13', 'rho23', 'sig_mean', 'rho', 'r', 'BARR_1/100', 'STRK_1_last/100', 'STRK_1_first/100', 'ANL_RTRN/100', 'TENOR', 'nobs', 'cpn_spread', 'b_over_k', 'stepdown', 'mom6m', 'ACT_ISU_AMT', 'SB_RT', 'DV_RT', 'PRCP_GRTE_RT', 'KNCK_IN_GRC_PRD', 'ISU_DT_year', 'SUB_END_DT-SUB_START_DT', 'recent_margin', 'recent_mktvol', 'curve_level', 'curve_slope', 'curve_curv', 'issue_intensity', 'ISU_ORG', 'RISK_GRADE', 'PRODUCT_TYPE', 'RDMP_TYPE', 'ISU_DT_month', 'ISU_DT_ordinal', 'FAIR_VALUE/ISU_PRC_DETAIL', 'MC']


,sig1,sig2,sig3,rho12,rho13,rho23,sig_mean,rho,r,BARR_1/100,...,curve_curv,issue_intensity,ISU_ORG,RISK_GRADE,PRODUCT_TYPE,RDMP_TYPE,ISU_DT_month,ISU_DT_ordinal,FAIR_VALUE/ISU_PRC_DETAIL,MC
0,0.112518,0.175696,0.188566,0.011789,0.183972,0.411278,0.158927,0.202347,0.010356,0.55,...,-0.006743,0,111722,2.0,ELS,DOWN,1,735603,0.900983,1.007757
1,0.115004,0.175696,0.188566,0.161165,0.183492,0.527834,0.159755,0.290830,0.010381,0.55,...,-0.006743,0,111722,2.0,ELS,DOWN,1,735603,0.889426,1.017281
2,0.112518,0.175696,0.188566,0.011789,0.183972,0.411278,0.158927,0.202347,0.010356,0.55,...,-0.006743,0,110893,2.0,ELS,DOWN,1,735603,0.914269,1.012476



=== pi_deeponet.csv | shape (23151, 37) ===
columns: ['sig1', 'sig2', 'sig3', 'rho12', 'rho13', 'rho23', 'sig_eff', 'u0', 'u1', 'u2', 'u3', 'u4', 'u5', 'u6', 'u7', 'u8', 'u9', 'strk_0', 'strk_1', 'strk_2', 'strk_3', 'strk_4', 'strk_5', 'strk_6', 'strk_7', 'strk_8', 'strk_9', 'strk_10', 'strk_11', 'BARR_1/100', 'ANL_RTRN/100', 'TENOR', 'r', 'ISU_DT_ordinal', 'FAIR_VALUE/ISU_PRC_DETAIL', 'MC', 'recent_margin']


,sig1,sig2,sig3,rho12,rho13,rho23,sig_eff,u0,u1,u2,...,strk_10,strk_11,BARR_1/100,ANL_RTRN/100,TENOR,r,ISU_DT_ordinal,FAIR_VALUE/ISU_PRC_DETAIL,MC,recent_margin
0,0.112518,0.175696,0.188566,0.011789,0.183972,0.411278,0.213083,-0.000056,0.001012,0.003133,...,0.7,0.7,0.55,0.052,2.992471,0.010356,735603,0.900983,1.007757,0.0
1,0.115004,0.175696,0.188566,0.161165,0.183492,0.527834,0.208856,-0.000056,0.001012,0.003133,...,0.8,0.8,0.55,0.070,3.000684,0.010381,735603,0.889426,1.017281,0.0
2,0.112518,0.175696,0.188566,0.011789,0.183972,0.411278,0.213083,-0.000056,0.001012,0.003133,...,0.7,0.7,0.55,0.060,2.992471,0.010356,735603,0.914269,1.012476,0.0



=== deeponet.csv | shape (23151, 37) ===


columns: ['u0', 'u1', 'u2', 'u3', 'u4', 'u5', 'u6', 'u7', 'u8', 'u9', 'sig1', 'sig2', 'sig3', 'rho12', 'rho13', 'rho23', 'sig_eff', 'strk_0', 'strk_1', 'strk_2', 'strk_3', 'strk_4', 'strk_5', 'strk_6', 'strk_7', 'strk_8', 'strk_9', 'strk_10', 'strk_11', 'BARR_1/100', 'ANL_RTRN/100', 'TENOR', 'r', 'ISU_DT_ordinal', 'FAIR_VALUE/ISU_PRC_DETAIL', 'MC', 'recent_margin']


,u0,u1,u2,u3,u4,u5,u6,u7,u8,u9,...,strk_10,strk_11,BARR_1/100,ANL_RTRN/100,TENOR,r,ISU_DT_ordinal,FAIR_VALUE/ISU_PRC_DETAIL,MC,recent_margin
0,-0.000056,0.001012,0.003133,0.005166,0.007063,0.010379,0.013064,0.0152,0.018244,0.020926,...,0.7,0.7,0.55,0.052,2.992471,0.010356,735603,0.900983,1.007757,0.0
1,-0.000056,0.001012,0.003133,0.005166,0.007063,0.010379,0.013064,0.0152,0.018244,0.020926,...,0.8,0.8,0.55,0.070,3.000684,0.010381,735603,0.889426,1.017281,0.0
2,-0.000056,0.001012,0.003133,0.005166,0.007063,0.010379,0.013064,0.0152,0.018244,0.020926,...,0.7,0.7,0.55,0.060,2.992471,0.010356,735603,0.914269,1.012476,0.0
